[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/juansensio/axr/blob/master/axr/segundo_parcial_conecta4.ipynb)

**Estados**: Cada configuración del tablero 4×4 es un estado. Se representa como un array numpy de 16 elementos con valores 1 (jugador 1), -1 (jugador 2) y 0 (vacío).

**Entorno**: El tablero 4×4 y las reglas del juego. El jugador elige una columna (0-3) y la ficha cae a la posición más baja disponible. El entorno devuelve el nuevo estado después de cada jugada.

**Política**: Es la estrategia que usa el agente para elegir qué columna jugar. Usamos el algoritmo de gradiente donde cada acción en cada estado tiene una preferencia $H(s,a)$. La probabilidad de elegir la acción $a$ en el estado $s$ es:

$$
\pi(s,a) = \frac{e^{H(s,a)}}{\sum_{b} e^{H(s,b)}}
$$

**Recompensa**: +1 para el ganador, 0 para el perdedor, 0.5 para cada uno en caso de empate.

**Función de acción (preferencia)**: $H(s,a)$ representa la preferencia por tomar la acción $a$ en el estado $s$. No es un valor de acción $Q(s,a)$, sino una preferencia relativa que se actualiza mediante ascenso por gradiente.

### Algoritmo de Gradiente

En lugar de estimar valores de acciones, el algoritmo de gradiente asigna preferencias $H(s,a)$. La probabilidad de seleccionar cada acción se obtiene con softmax:

$$
\pi(s,a) = \frac{e^{H(s,a)}}{\sum_{b} e^{H(s,b)}}
$$

Al final de cada partida, las preferencias se actualizan:

$$
\begin{array}{ll}
H(s,a) = H(s,a) + \alpha(R - \bar{R})(1 - \pi(s,a)) & \text{si } a = A_t \\
H(s,a) = H(s,a) - \alpha(R - \bar{R})\pi(s,a) & \text{para } a \neq A_t
\end{array}
$$

donde $R$ es la recompensa total de la partida, $\bar{R}$ es el promedio de todas las recompensas y $\alpha$ es la tasa de aprendizaje.

**Ejemplo**: Si en un estado $s$ hay 3 columnas disponibles y el agente elige la columna 2 ganando la partida ($R=1$), entonces $H(s,2)$ aumenta y $H(s,0)$, $H(s,1)$ disminuyen. Así el agente aprende a repetir jugadas ganadoras.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import math
from tqdm import tqdm

In [ ]:
class Board():
    def __init__(self):
        self.state = np.zeros((4, 4))

    def valid_moves(self):
        return [c for c in range(4) if self.state[0, c] == 0]

    def update(self, symbol, col):
        if col < 0 or col >= 4 or self.state[0, col] != 0:
            raise ValueError("movimiento ilegal")
        for r in range(3, -1, -1):
            if self.state[r, col] == 0:
                self.state[r, col] = symbol
                break

    def is_game_over(self):
        for r in range(4):
            if abs(self.state[r].sum()) == 4:
                return 1 if self.state[r].sum() > 0 else -1
        for c in range(4):
            if abs(self.state[:, c].sum()) == 4:
                return 1 if self.state[:, c].sum() > 0 else -1
        d1 = sum([self.state[i, i] for i in range(4)])
        if abs(d1) == 4:
            return 1 if d1 > 0 else -1
        d2 = sum([self.state[i, 3-i] for i in range(4)])
        if abs(d2) == 4:
            return 1 if d2 > 0 else -1
        if len(self.valid_moves()) == 0:
            return 0
        return None

    def reset(self):
        self.state = np.zeros((4, 4))

In [ ]:
def softmax(x):
    e_x = np.exp(x - np.max(x))
    return e_x / e_x.sum()


class Agent():
    def __init__(self, alpha=0.1):
        self.H = {}
        self.alpha = alpha
        self.positions = []
        self.actions = []
        self.recompensas = []

    def reset(self):
        self.positions = []
        self.actions = []

    def move(self, board, explore=True):
        valid_moves = board.valid_moves()
        s = str(board.state.reshape(16))
        n = len(valid_moves)
        if s not in self.H:
            self.H[s] = np.zeros(n)
        elif len(self.H[s]) != n:
            tmp = self.H[s].copy()
            self.H[s] = np.zeros(n)
            self.H[s][:len(tmp)] = tmp[:min(len(tmp), n)]
        pi = softmax(self.H[s])
        ix = np.random.choice(range(n), p=pi)
        self.positions.append(s)
        self.actions.append(ix)
        return valid_moves[ix]

    def update(self, board):
        pass

    def reward(self, reward):
        self.recompensas.append(reward)
        recompensa_media = np.mean(self.recompensas)
        for i in reversed(range(len(self.positions))):
            s = self.positions[i]
            a = self.actions[i]
            if s not in self.H:
                continue
            n = len(self.H[s])
            pi = softmax(self.H[s])
            for j in range(n):
                if j == a:
                    self.H[s][j] += self.alpha * (reward - recompensa_media) * (1 - pi[j])
                else:
                    self.H[s][j] -= self.alpha * (reward - recompensa_media) * pi[j]

In [ ]:
class Game():
    def __init__(self, player1, player2):
        player1.symbol = 1
        player2.symbol = -1
        self.players = [player1, player2]
        self.board = Board()

    def selfplay(self, rounds=100):
        wins = [0, 0]
        for i in tqdm(range(1, rounds + 1)):
            self.board.reset()
            for player in self.players:
                player.reset()
            game_over = False
            while not game_over:
                for player in self.players:
                    action = player.move(self.board)
                    self.board.update(player.symbol, action)
                    for player in self.players:
                        player.update(self.board)
                    if self.board.is_game_over() is not None:
                        game_over = True
                        break
            self.reward()
            for ix, player in enumerate(self.players):
                if self.board.is_game_over() == player.symbol:
                    wins[ix] += 1
        return wins

    def reward(self):
        winner = self.board.is_game_over()
        if winner == 0:
            for player in self.players:
                player.reward(0.5)
        else:
            for player in self.players:
                if winner == player.symbol:
                    player.reward(1)
                else:
                    player.reward(0)

### Entrenamiento del agente

Entrenamos dos agentes enfrentándolos entre sí usando el algoritmo de gradiente con diferentes tasas de aprendizaje $\alpha$.

In [ ]:
alphas = [0.05, 0.1, 0.4]
partidas = 100
ventana = 100
tasas_exito = np.zeros((len(alphas), partidas // ventana))

for i, alpha in enumerate(alphas):
    agente1 = Agent(alpha=alpha)
    agente2 = Agent(alpha=alpha)
    game = Game(agente1, agente2)
    wins_agente1 = 0
    for ep in range(1, partidas + 1):
        wins = game.selfplay(rounds=1)
        wins_agente1 += wins[0]
        if ep % ventana == 0:
            tasas_exito[i][ep // ventana - 1] = wins_agente1 / ventana
            wins_agente1 = 0
            print(f'α={alpha}: ventana {ep//ventana}/{(partidas//ventana)} - tasa éxito: {tasas_exito[i][ep//ventana-1]:.3f}')

In [ ]:
plt.figure(figsize=(8,5))
for i, a in enumerate(alphas):
    plt.plot(range(1, partidas // ventana + 1), tasas_exito[i], marker='o', label=fr'$\\alpha$ = {a}')
plt.axhline(y=0.5, color='gray', linestyle='--', label='Aleatorio')
plt.legend()
plt.grid(True)
plt.xlabel('Ventana (cada 1000 partidas)')
plt.ylabel('Tasa de éxito del agente 1')
plt.title('Algoritmo de Gradiente en Conecta 4 (4×4)')
plt.show()